# Listen to the corpus and explore the rhythms

This notebook downloads three takes of the IEMP Malian Jembe corpus from OSF into **your** Colab session, then lets you:

1. listen to each piece and to its dundun against an even 12-pulse grid, to hear the micro-timing;
2. see the timing analysis behind it;
3. open the *Boussole des rythmes*, an interactive page that draws each rhythm on a 12-pulse circle and resynthesises it.

The corpus (Polak, Tarsitani & Clayton, 2018, OSF `M652X`) is CC BY-NC-ND 4.0. Audio is fetched and played only inside this session; nothing is saved to the repository. Clear the outputs before saving a copy of this notebook anywhere public.

Runtime: CPU is enough. The download is about 330 MB and takes a minute or two on Colab.

## 1. Setup
If pip reports that numpy was replaced, use *Runtime > Restart session*, then continue from the next cell.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
REPO = "/content/Djembe" if IN_COLAB else os.path.abspath("..")
COMMIT = "ee54db53252ba9ced53dbaaa6a668598725d8b68"   # pinned code version; bump it when the code changes
if IN_COLAB and not os.path.exists(REPO):
    !git clone -q https://github.com/clementdumeril/Djembe.git "{REPO}" && git -C "{REPO}" checkout -q {COMMIT}
%pip install -q -r "{REPO}/requirements-demo.txt"

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
REPO = "/content/Djembe" if IN_COLAB else os.path.abspath("..")
sys.path.insert(0, REPO); os.chdir(REPO)
CORPUS = "/content/corpus" if IN_COLAB else os.path.join(REPO, "corpus")

import djembe_dataset as d
TAKES = ["Manjanin_1", "Maraka_6", "Woloso_2"]   # the takes with both dunun parts recorded
d.download_corpus(CORPUS, audio=True, audio_takes=TAKES)
events = d.build_events(CORPUS)
print(len(events), "annotated strokes")

## 2. Listen
For each piece: 15 seconds of the ensemble, then the same 15 seconds of dundun 1 with a click on an even 12-pulse grid (higher click on each cycle start). Where a stroke lands before or after its click is the micro-timing measured in section 3.

In [ ]:
from IPython.display import Audio, Markdown, display
import demo_audio

for take in TAKES:
    piece = take.split("_")[0]
    display(Markdown(f"**{piece}** ({take}): ensemble"))
    display(Audio(demo_audio.ensemble_excerpt(CORPUS, take), rate=demo_audio.SR_OUT))
    display(Markdown("dundun 1 against the even grid"))
    display(Audio(demo_audio.dundun_against_grid(CORPUS, take), rate=demo_audio.SR_OUT))

## 3. Pattern and micro-timing
The same figure as in the README, computed here from the annotations: where dundun 1 plays in the cycle, and how far each stroke sits from the even grid.

In [ ]:
import make_figures
from IPython.display import Image
make_figures.FIG_DIR = make_figures.Path(REPO) / "output" / "demo_figures"   # git-ignored
make_figures.FIG_DIR.mkdir(parents=True, exist_ok=True)
Image(str(make_figures.hero(events)))

In [ ]:
# Mean offset of each instrument per position in the beat (fraction of a pulse).
events.dropna(subset=["dev_sd"]).assign(position_in_beat=lambda e: (e.sd - 1) % 3 + 1)       .pivot_table(index=["piece", "instrument"], columns="position_in_beat", values="dev_sd", aggfunc="mean").round(3)

## 4. Boussole des rythmes
The page is built in this session from the annotations and the downloaded takes, then served from the notebook. Use *Lancer la boucle* to hear the resynthesis; *Enregistrement* plays the takes you downloaded.

In [ ]:
import utils_V2, boussole_rythmes_V2 as boussole
corpus_link = os.path.join(REPO, "corpus")
if os.path.abspath(CORPUS) != os.path.abspath(corpus_link) and not os.path.exists(corpus_link):
    os.symlink(CORPUS, corpus_link)          # the page expects corpus/ next to output/
path, size = boussole.build()
print(path, size // 1000, "kB")

if IN_COLAB:
    import subprocess
    from google.colab import output
    subprocess.Popen([sys.executable, "-m", "http.server", "8000", "--bind", "127.0.0.1"], cwd=REPO,
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    output.serve_kernel_port_as_iframe(8000, path="/output/boussole_rythmes.html", height=1000)
else:
    print("Open it with: python -m http.server 8000, then http://127.0.0.1:8000/output/boussole_rythmes.html")